In [ ]:
import os
import re
import warnings
import pandas as pd
import numpy as np
import torch
from collections import Counter
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EvalPrediction
)
from sklearn.metrics import f1_score, roc_auc_score, jaccard_score
import wandb

warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}  |  GPU: {torch.cuda.is_available()}')

c:\Users\abhay\Desktop\projects\movie-classification\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 1.12.1+cu113  |  GPU: True


In [ ]:

# --- LOCAL ---
DATASET_PATH = '../dataset'

# --- COLAB: uncomment below and comment out the line above ---
# import kagglehub
# kag_path = kagglehub.dataset_download("rajugc/imdb-movies-dataset-based-on-genre")
# DATASET_PATH = kag_path

In [ ]:
# Same approach as the other notebooks: read each CSV file, combine, deduplicate

frames = []
for filename in os.listdir(DATASET_PATH):
    if filename.endswith('.csv'):
        filepath = os.path.join(DATASET_PATH, filename)
        df_temp = pd.read_csv(filepath, usecols=['movie_id', 'movie_name', 'description', 'genre'])
        frames.append(df_temp)

df = pd.concat(frames, ignore_index=True)
print(f'Total rows after combining: {df.shape[0]}')

# Drop rows missing description or genre
df = df.dropna(subset=['description', 'genre'])

# Parse genre strings into lists early so we can aggregate them
df['genre'] = df['genre'].apply(lambda x: [g.strip() for g in x.split(',')] if isinstance(x, str) else [])

# Group by movie_id: a movie appears once per genre file, so we merge its genre lists
df = df.groupby('movie_id').agg(
    movie_name=('movie_name', 'first'),
    description=('description', 'first'),
    genre=('genre', lambda x: sorted(set(g for row in x for g in row)))
).reset_index()

# Fill missing movie names
df['movie_name'] = df['movie_name'].fillna('Unknown')

# Keep only rows where description has at least 10 words
df = df[df['description'].str.split().str.len() >= 10].reset_index(drop=True)

print(f'After cleaning: {df.shape[0]} movies')
df.head(3)

Total rows after combining: 368300
After cleaning: 181735 movies


,movie_id,movie_name,description,genre
0,tt0000679,The Fairylogue and Radio-Plays,L. Frank Baum would appear in a white suit and...,"[Adventure, Fantasy]"
1,tt0001115,Ansigttyven I,Consul Bjørn is urgently called to a company m...,[Crime]
2,tt0001175,La dame aux camélias,Marguerite is a courtesan in Paris. She falls ...,"[Drama, Romance]"


In [ ]:
MIN_GENRE_COUNT = 1000

# Count how many movies each genre appears in
all_genres_flat = []
for genre_list in df['genre']:
    for genre in genre_list:
        all_genres_flat.append(genre)

genre_counts = pd.Series(Counter(all_genres_flat)).sort_values(ascending=False)

# Keep only genres that appear in at least 1000 movies
GENRE_COLS = sorted([g for g, c in genre_counts.items() if c >= MIN_GENRE_COUNT])
print(f'Keeping {len(GENRE_COLS)} genres: {GENRE_COLS}')

# Remove genres from each movie's list that are not in the valid set
df['genre'] = df['genre'].apply(lambda gs: [g for g in gs if g in GENRE_COLS])

# Remove movies that have no valid genres left
df = df[df['genre'].apply(len) > 0].reset_index(drop=True)

print(f'Final dataset: {df.shape[0]:,} movies | {len(GENRE_COLS)} genres')

Keeping 20 genres: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Sport', 'Thriller', 'War', 'Western']
Final dataset: 181,735 movies | 20 genres


In [ ]:
# Same as notebook 02 — fold 0 gives us our 80/20 train/test split

X = df[['description']]

# Build a binary matrix for stratification purposes
y_rows = []
for genre_list in df['genre']:
    row = [1 if g in genre_list else 0 for g in GENRE_COLS]
    y_rows.append(row)
y_dummy = pd.DataFrame(y_rows, columns=GENRE_COLS)

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(mskf.split(X, y_dummy))

# Keep only description and genre columns — that is all the model needs
train_df = df.iloc[train_idx][['description', 'genre']].reset_index(drop=True)
test_df  = df.iloc[test_idx][['description', 'genre']].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

Train: 145,405  |  Test: 36,330


In [ ]:

train_hf = Dataset.from_pandas(train_df)
test_hf  = Dataset.from_pandas(test_df)
movie_dataset = DatasetDict({'train': train_hf, 'test': test_hf})
print(movie_dataset)

DatasetDict({
    train: Dataset({
        features: ['description', 'genre'],
        num_rows: 145405
    })
    test: Dataset({
        features: ['description', 'genre'],
        num_rows: 36330
    })
})


In [ ]:
# The model needs to know: index -> genre name, and genre name -> index

id2label = {}
for i, label in enumerate(GENRE_COLS):
    id2label[i] = label

label2id = {}
for i, label in enumerate(GENRE_COLS):
    label2id[label] = i

NUM_LABELS = len(GENRE_COLS)

print(f'NUM_LABELS: {NUM_LABELS}')
print(id2label)

NUM_LABELS: 20
{0: 'Action', 1: 'Adventure', 2: 'Animation', 3: 'Biography', 4: 'Comedy', 5: 'Crime', 6: 'Drama', 7: 'Family', 8: 'Fantasy', 9: 'History', 10: 'Horror', 11: 'Music', 12: 'Musical', 13: 'Mystery', 14: 'Romance', 15: 'Sci-Fi', 16: 'Sport', 17: 'Thriller', 18: 'War', 19: 'Western'}


In [ ]:
#
# Bug fix: the original version compared genre strings against Arrow sequences,
# which failed silently (all labels became 0.0). The fix is to explicitly loop
# through each example's genre list and build a plain Python float list.

def make_preprocess_fn(tokenizer):
    def preprocess(examples):
        # Tokenize the description text — padding is handled by DataCollatorWithPadding
        encoding = tokenizer(
            examples['description'],
            truncation=True,
            max_length=256,
            padding=False
        )

        # Build the multi-hot label vector for each example in the batch
        # genre_list is a list of genre strings for one movie (e.g. ['Action', 'Comedy'])
        labels = []
        for genre_list in examples['genre']:
            row = []
            for label in GENRE_COLS:
                if label in genre_list:
                    row.append(1.0)
                else:
                    row.append(0.0)
            labels.append(row)

        encoding['labels'] = labels
        return encoding

    return preprocess

In [ ]:
# Called by the Trainer after each eval epoch
# Applies sigmoid to raw logits, then thresholds at 0.5 to get binary predictions

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions

    # Convert logits to probabilities
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.tensor(preds)).numpy()

    # Apply threshold — anything above 0.5 counts as a predicted genre
    y_pred = (probs >= 0.5).astype(int)
    y_true = p.label_ids.astype(int)

    return {
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'jaccard':  jaccard_score(y_true, y_pred, average='micro', zero_division=0),
        'roc_auc':  roc_auc_score(y_true, probs, average='micro'),
    }

In [10]:
# Training function
# batch_size=8 to fit within 4GB VRAM on GTX 1650
# save_strategy='epoch' saves a checkpoint after every epoch
# save_total_limit=2 keeps only 2 most recent checkpoints to save disk space
# If training crashes, resume by calling trainer.train(resume_from_checkpoint=True)

def train_model(model_name, output_dir, dataset):
    print(f'Loading tokenizer: {model_name}')
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    preprocess_fn = make_preprocess_fn(tokenizer)
    tokenized = dataset.map(
        preprocess_fn,
        batched=True,
        batch_size=128,
        remove_columns=['genre', 'description']
    )

    print(f'Loading model: {model_name}')
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        problem_type='multi_label_classification',
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    )

    args = TrainingArguments(
        output_dir=output_dir,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=32,
        num_train_epochs=5,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='jaccard',
        greater_is_better=True,
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to='wandb',
        run_name=output_dir,
    )

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['test'],
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    print(f'Starting training - {args.num_train_epochs} epochs...')
    trainer.train()
    return trainer, tokenizer

In [11]:
# Load API keys from .env file — fill in your keys in .env before running
# .env is in .gitignore so it never goes to GitHub
import os
from dotenv import load_dotenv

load_dotenv('../.env')  # loads WANDB_API_KEY and HF_TOKEN into os.environ

WANDB_KEY = os.environ.get('WANDB_API_KEY', '')
if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
    print('W&B login successful')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('No WANDB_API_KEY in .env — W&B disabled. Training will still run.')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\abhay\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B login successful


In [12]:
# Cell 13 — Train DistilBERT
# DistilBERT is 40% smaller than BERT, runs faster, and still performs well

DISTILBERT = 'distilbert-base-uncased'
distilbert_trainer, distilbert_tokenizer = train_model(
    DISTILBERT, 'distilbert-genre', movie_dataset
)

distilbert_eval = distilbert_trainer.evaluate()
print('DistilBERT eval results:')
print(distilbert_eval)

Loading tokenizer: distilbert-base-uncased


Map: 100%|██████████| 36330/36330 [00:01<00:00, 18875.45 examples/s]


Loading model: distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training - 5 epochs...


...

{'eval_loss': 0.20802980661392212, 'eval_f1_micro': 0.5358693934079499, 'eval_f1_macro': 0.4561366392738505, 'eval_jaccard': 0.3659983549249434, 'eval_roc_auc': 0.9142562887037463, 'eval_runtime': 702.8678, 'eval_samples_per_second': 51.688, 'eval_steps_per_second': 1.616, 'epoch': 5.0}


100%|██████████| 22720/22720 [18:16:48<00:00,  2.90s/it]


{'train_runtime': 65812.1985, 'train_samples_per_second': 11.047, 'train_steps_per_second': 0.345, 'train_loss': 0.19606714487915308, 'epoch': 5.0}


100%|██████████| 1136/1136 [11:59<00:00,  1.58it/s]

DistilBERT eval results:
{'eval_loss': 0.20802980661392212, 'eval_f1_micro': 0.5358693934079499, 'eval_f1_macro': 0.4561366392738505, 'eval_jaccard': 0.3659983549249434, 'eval_roc_auc': 0.9142562887037463, 'eval_runtime': 721.1041, 'eval_samples_per_second': 50.381, 'eval_steps_per_second': 1.575, 'epoch': 5.0}


In [ ]:
# Final comparison table
results = pd.DataFrame([
    {
        'Model': 'TF-IDF + Logistic Regression',
        'Micro F1': 0.4709,
        'Macro F1': 0.4032,
        'Jaccard':  0.3079,
        'ROC-AUC':  0.7717,
    },
    {
        'Model': 'DistilBERT-base-uncased',
        'Micro F1': round(distilbert_eval['eval_f1_micro'], 4),
        'Macro F1': round(distilbert_eval['eval_f1_macro'], 4),
        'Jaccard':  round(distilbert_eval['eval_jaccard'], 4),
        'ROC-AUC':  round(distilbert_eval['eval_roc_auc'], 4),
    },
])

print(results.to_string(index=False))

In [ ]:
# Push best model to Hugging Face Hub
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv('../.env')

HF_USERNAME = 'Abhay-learns'
HF_REPO = f'{HF_USERNAME}/distilbert-genre'

hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    login(token=hf_token)
    print('HuggingFace login successful')
else:
    print('WARNING: HF_TOKEN not in .env — model will be saved locally only.')

# Only DistilBERT was trained — use it directly
best_trainer = distilbert_trainer
best_model_name = 'DistilBERT-base-uncased'
print(f'Best model: {best_model_name}')

# Always save locally first
best_trainer.save_model('best-model-final')
print('Model saved locally to: best-model-final/')

# Push to HF Hub
if hf_token:
    best_trainer.push_to_hub(HF_REPO)
    print(f'Model pushed to: https://huggingface.co/{HF_REPO}')
else:
    print('Set HF_TOKEN in .env and re-run this cell to push.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace login successful
Best model: DistilBERT-base-uncased
Model saved locally to: best-model-final/


Processing Files (2 / 2): 100%|██████████|  268MB /  268MB, 4.49MB/s  
New Data Upload: 100%|██████████|  268MB /  268MB, 4.49MB/s  


Model pushed to: https://huggingface.co/Abhay-learns/movie-genre-classifier
